In [109]:
import glob
import os
import time
import tifffile as tiff
from scipy import ndimage as ndi
from skimage import morphology
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


In [110]:
def test():
    input = r"C:\Users\morit\Desktop\work\study\Watershed\Data\Input\20260313_test\label_map_1_annotate.tif"
    frames = tiff.imread(input)
    frame = frames[100]
    structure = ndi.generate_binary_structure(rank=2, connectivity=2)
    labels, labels_number = ndi.label(frame, structure=structure)
    tiff.imwrite(r"C:\Users\morit\Desktop\work\study\Watershed\Data\output\20260606\test.tif", labels.astype(np.uint16), dtype=np.uint16)



In [111]:
# グラフ以外の処理
def load_volume(input_path: str):
    return tiff.imread(input_path)
import numpy as np
import tifffile as tiff

def zero_slices_around_local_minima(volume: np.ndarray, half_window: int = 5):
    # 2値volumeの面積プロファイルを見て、局所極小の前後half_windowスライスを0化する
    area_profile = np.count_nonzero(volume, axis=(1, 2))
    minima_indices = []

    for index in range(1, len(area_profile) - 1):
        prev_area = area_profile[index - 1]
        curr_area = area_profile[index]
        next_area = area_profile[index + 1]
        if curr_area <= prev_area and curr_area <= next_area and (curr_area < prev_area or curr_area < next_area):
            minima_indices.append(index)

    output_volume = volume.copy()
    zeroed_ranges = []
    for index in minima_indices:
        start = max(0, index - half_window)
        end = min(volume.shape[0], index + half_window + 1)
        output_volume[start:end] = 0
        zeroed_ranges.append((start + 1, end))  # 1始まりで記録

    return output_volume, area_profile, minima_indices, zeroed_ranges

# 使い方の例: 既に volume がある場合はそれを使う
# volume = load_volume(r"D:\\_study\\ImageProcessing\\study\\Watershed\\Data\\Input\\labeled_map_filled\\label_map_3_filled.tif")
# zeroed_volume, area_profile, minima_indices, zeroed_ranges = zero_slices_around_local_minima(volume, half_window=5)
# print(f"local minima slices (1-based): {[i + 1 for i in minima_indices]}")
# print(f"zeroed ranges (1-based): {zeroed_ranges}")
# tiff.imwrite(r"D:\\_study\\ImageProcessing\\study\\Watershed\\Data\\output\\20260610_minima_zeroed\\volume_minima_zeroed.tif", zeroed_volume.astype(np.uint16), dtype=np.uint16)

In [112]:
# 重み付きグラフの作成
def create_graph(volume: int, threshold: int, labels_connectivity: int =2):
    G = nx.Graph()
    labeled_volume = np.zeros_like(volume, dtype=np.uint16)
    for index, curr_slice in enumerate(volume):
        # スライスは1始まりにする
        slice_index = index + 1
        debug = False
        if debug and (slice_index >= 379 or slice_index <= 330):
            continue
        structure = ndi.generate_binary_structure(rank=2, connectivity=labels_connectivity)
        labels, labels_number = ndi.label(curr_slice, structure=structure)
        for label_number in range(1, labels_number+1):
            label_area = np.sum(labels == label_number)
            # 面積が重み
            G.add_node((slice_index, label_number), area = label_area, seed = False)
        if slice_index == 1:
            continue
        # 直前のスライス
        prev_slice_index = slice_index - 1
        prev_slice = volume[prev_slice_index - 1]
        prev_labels, prev_labels_number = ndi.label(prev_slice, structure=structure)
        # 重なりが0でない場所にエッジを引く
        for label_number in range(1, labels_number+1):
            overlap_prev_label_numbers = np.unique(prev_labels[labels == label_number])
            overlap_prev_label_numbers = overlap_prev_label_numbers[overlap_prev_label_numbers != 0]
            # overlap_prev_label_numbers >= 2: 重なりが2つかつ下位ノード2つが閾値以上ならエッジを引かない
            
            edge_color = "blue" # まともなエッジは青
            # len が 1 以下か、閾値以下のノードならまとも扱い
            edge_color_count = 0
            for overlap_prev_label_number in overlap_prev_label_numbers:
                if len(overlap_prev_label_numbers) >= 2 and \
                    threshold <= G.nodes[(prev_slice_index, overlap_prev_label_number)]["area"]:
                    edge_color_count += 1
            if edge_color_count >= len(overlap_prev_label_numbers):
                # すべて閾値以上なら切断っ候補
                edge_color = "red" # 切断候補は赤
            for overlap_prev_label_number in overlap_prev_label_numbers:
                overlap_prev_label_number = int(overlap_prev_label_number)
                if ((slice_index, label_number) in G)\
                and ((prev_slice_index, overlap_prev_label_number)) in G:
                    G.add_edge(
                        (slice_index, label_number), 
                        (prev_slice_index, overlap_prev_label_number),
                        color = edge_color
                    )
                else:
                    # debug で最下層のノードはエッジが引けないため
                    # print(f"    Not Exists: {(slice_index, label_number)} or {(slice_index-1, overlap_prev_label_number)}")
                    continue
        
        # スライスごとにラベリングしたものを保存しておく
        labeled_volume[index] = labels.astype(np.uint16)

        # エッジ削除
        # slice_index-1 の接続を見て、つながっているノード(slice_index) 

        # 表示用
        G.add_node((slice_index, 'slice_index'), seed=False)
    return G, labeled_volume

def remove_edge(G: nx):
    # 赤いエッジでそれ以下のsliceに赤エッジがあれば緑にする
    # 赤エッジのslice以下の隣接エッジで
    # print("remove edge")
    count = 0
    for previous_node, current_node, attr in \
        sorted(G.edges(data=True),\
                key=lambda e: max(e[0][0], e[1][0]),\
                reverse=True):
        if attr.get("color") == "red":
            # print(f"find red edge: {previous_node} --- {current_node}")
            if (find_red_edge_previous_node(G, previous_node)):
                # 赤いエッジがあったので保留
                G.edges[current_node, previous_node]["color"] = "green"
            else:
                # 赤いエッジがないので切断する
                # G.remove_edge(current_node, previous_node)
                # 切断時の下位ノードをseedとする
                G.nodes[previous_node]["seed"] = True
                continue


def red_descendants_with_edges(G, start_node):
    # print(f"start node: {start_node}")
    visited = set()
    stack = [start_node]

    descendant_nodes = set()
    descendant_edges = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        # print(list(G.neighbors(node)))
        for neighbor in G.neighbors(node):
            # 子方向のみ
            if neighbor[0] <= node[0]:
                continue
            # 赤エッジのみ
            if G[node][neighbor].get("color") != "red":
                continue
            descendant_nodes.add(neighbor)
            descendant_edges.append((node, neighbor))
            stack.append(neighbor)
    return descendant_nodes, descendant_edges

def find_red_edge_previous_node(G, start_node):
    # start_node 以下の隣接スライスで赤エッジがないか判定する
    # return true: 赤エッジない -> こいつが seed
    visited = set() # 探索済みのノード
    stack = [start_node]

    descendant_nodes = set()
    descendant_edges = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node) # 探索済みノードに加える
        for neighbor_node in G.neighbors(node):
            # 子ノードのみ(slice_indexが小さい)探索
            if neighbor_node[0] > node[0]:
                continue
            # 赤エッジの有無を判定
            if G[node][neighbor_node].get("color") == "red":
                # print(f"            red edge: {node} to {neighbor_node}")
                return True
            # 下の層へ進むため、 neighbor_node を stack に入れる
            stack.append(neighbor_node)
    # 赤エッジが存在しないならfalse: 切断しない
    return False

def plot_graph(G: nx):
        # --- 描画 ---
    # 実ノード(ラベルノード)が存在するスライスだけ slice_index ノードを表示する
    slices_with_real_nodes = {
        node[0]
        for node in G.nodes
        if isinstance(node[1], (int, np.integer))
    }

    draw_nodes = [
        node
        for node in G.nodes
        if node[1] != 'slice_index' or node[0] in slices_with_real_nodes
    ]
    draw_graph = G.subgraph(draw_nodes).copy()

    pos = {}
    for node in draw_graph.nodes:
        slice_index, label = node
        if label == 'slice_index':
            pos[node] = (0, slice_index) # 左端
        else:
            pos[node] = (label, slice_index) # x:スライス順, y:ラベル番号で上下にずらす
    plt.figure(figsize=(12, 100))
    labels = {}
    for node in draw_graph.nodes:
        if node[1] == 'slice_index':
            labels[node] = f"slice {node[0]}"
        else:
            labels[node] = f"{draw_graph.nodes[node]['area']}"
            continue
    node_colors = []
    for node in draw_graph.nodes:
        if draw_graph.nodes[node].get("seed", True):
            node_colors.append("red")
        else:
            node_colors.append('skyblue')
    edge_colors = [
        draw_graph[u][v].get("color", "black")
        for u, v in draw_graph.edges
    ]

    nx.draw(draw_graph, pos=pos, 
            with_labels=True, 
            labels=labels, 
            node_color=node_colors,
            edge_color=edge_colors)

def create_seed_for_watershed(G: nx, labeled_volume: np.ndarray):
    # 赤ノード(slice, label)に一致する画素だけ値を残す
    seed_volume = np.zeros_like(labeled_volume, dtype=labeled_volume.dtype)
    red_nodes = [
        node
        for node, attr in G.nodes(data=True)
        if attr.get("seed", False)
    ]

    next_seed_label = 1
    for slice_index, label_number in sorted(red_nodes, key=lambda node: (node[0], node[1])):
        if not isinstance(label_number, (int, np.integer)):
            continue
        if not (1 <= slice_index <= labeled_volume.shape[0]):
            continue

        label_number = int(label_number)
        slice_labels = labeled_volume[slice_index - 1]
        mask = slice_labels == label_number
        if not np.any(mask):
            continue
        seed_volume[slice_index - 1][mask] = next_seed_label
        next_seed_label += 1

    return seed_volume

def merge_bidirectional_graph(forward_graph: nx.Graph, reverse_graph: nx.Graph, num_slices: int):
    # 反転側グラフのslice indexを元座標に戻して統合する
    merged_graph = nx.Graph()

    color_priority = {"black": 0, "blue": 1, "green": 2, "red": 3}

    def upsert_node(node, attrs):
        if node not in merged_graph:
            merged_graph.add_node(node, **attrs)
            return

        merged_attrs = merged_graph.nodes[node]
        merged_attrs["seed"] = merged_attrs.get("seed", False) or attrs.get("seed", False)
        if "area" in attrs:
            merged_attrs["area"] = max(int(merged_attrs.get("area", 0)), int(attrs["area"]))

    def upsert_edge(node_u, node_v, attrs):
        new_color = attrs.get("color", "black")
        if merged_graph.has_edge(node_u, node_v):
            current_color = merged_graph[node_u][node_v].get("color", "black")
            if color_priority.get(new_color, 0) > color_priority.get(current_color, 0):
                merged_graph[node_u][node_v]["color"] = new_color
            return
        merged_graph.add_edge(node_u, node_v, color=new_color)

    for node, attrs in forward_graph.nodes(data=True):
        upsert_node(node, dict(attrs))
    for node_u, node_v, attrs in forward_graph.edges(data=True):
        upsert_edge(node_u, node_v, attrs)

    for node, attrs in reverse_graph.nodes(data=True):
        mapped_node = (num_slices - node[0] + 1, node[1])
        upsert_node(mapped_node, dict(attrs))
    for node_u, node_v, attrs in reverse_graph.edges(data=True):
        mapped_u = (num_slices - node_u[0] + 1, node_u[1])
        mapped_v = (num_slices - node_v[0] + 1, node_v[1])
        upsert_edge(mapped_u, mapped_v, attrs)

    return merged_graph

def detect_non_bifurcating_components(G: nx.Graph):
    # 二股(同一ノードから次スライス方向へ2本以上)を持たない連結成分を抽出
    valid_nodes = [
        node
        for node in G.nodes
        if isinstance(node[1], (int, np.integer))
    ]
    subgraph = G.subgraph(valid_nodes)

    non_bifurcating_components = []
    for component_nodes in nx.connected_components(subgraph):
        component_nodes = list(component_nodes)
        has_bifurcation = False

        for node in component_nodes:
            children = [
                neighbor
                for neighbor in subgraph.neighbors(node)
                if neighbor[0] > node[0]
            ]
            if len(children) >= 2:
                has_bifurcation = True
                break

        if not has_bifurcation:
            non_bifurcating_components.append(
                sorted(component_nodes, key=lambda n: (n[0], n[1]))
            )

    return non_bifurcating_components

def add_largest_area_seed_in_components(G: nx.Graph, components):
    # 各検出成分でseed未確定なら、面積最大ノードをseedへ追加する
    added_seed_nodes = []
    for component_nodes in components:
        if not component_nodes:
            continue

        has_seed = any(G.nodes[node].get("seed", False) for node in component_nodes)
        if has_seed:
            continue

        target_node = max(
            component_nodes,
            key=lambda n: int(G.nodes[n].get("area", 0))
        )
        G.nodes[target_node]["seed"] = True
        added_seed_nodes.append(target_node)
    return added_seed_nodes



def graph_main(input_path: str, inverse_volume: bool = True):
    volume = load_volume(input_path)
    # volume, _, _, _ = zero_slices_around_local_minima(volume, half_window=5)
    threshold = 1000
    G_forward, labeled_volume = create_graph(volume, threshold)
    remove_edge(G_forward)

    G_reverse = None
    labeled_volume_reverse = None
    merged_graph = G_forward

    if inverse_volume:
        volume_reverse = volume[::-1]
        G_reverse, labeled_volume_reverse = create_graph(volume_reverse, threshold)
        remove_edge(G_reverse)
        merged_graph = merge_bidirectional_graph(G_forward, G_reverse, volume.shape[0])

    non_bifurcating_components = detect_non_bifurcating_components(merged_graph)
    added_seed_nodes = add_largest_area_seed_in_components(merged_graph, non_bifurcating_components)
    # print(f"non-bifurcating components: {len(non_bifurcating_components)}")
    # print(f"added largest-area seeds: {len(added_seed_nodes)}")

    # 追加seedを順転・反転グラフへ反映してseed volumeを再生成
    for node in added_seed_nodes:
        if node in G_forward:
            G_forward.nodes[node]["seed"] = True
        if inverse_volume and G_reverse is not None:
            reverse_node = (volume.shape[0] - node[0] + 1, node[1])
            if reverse_node in G_reverse:
                G_reverse.nodes[reverse_node]["seed"] = True

    seed_volume_forward = create_seed_for_watershed(G_forward, labeled_volume)
    seed_volume = seed_volume_forward
    if inverse_volume and G_reverse is not None and labeled_volume_reverse is not None:
        seed_volume_reverse = create_seed_for_watershed(G_reverse, labeled_volume_reverse)
        seed_volume_reverse = seed_volume_reverse[::-1]
        seed_volume = np.where(seed_volume_forward != 0, seed_volume_forward, seed_volume_reverse)

    # tiff.imwrite(r"D:\_study\ImageProcessing\study\Watershed\Data\output\20260609\seed_volume_1.tif", seed_volume.astype(np.uint16), dtype=np.uint16)
    # plot_graph(merged_graph)

    return merged_graph, seed_volume


In [113]:
def apply_seed_slice_ranges(
    seed_volume: np.ndarray,
    upper_slice_range: tuple | None = None,
    lower_slice_range: tuple | None = None,
) -> np.ndarray:
    # 1-based inclusive なスライス範囲指定で seed を残す
    keep_mask = np.zeros(seed_volume.shape[0], dtype=bool)

    def _mark_slice_range(slice_range: tuple | None):
        if slice_range is None:
            return
        if len(slice_range) != 2:
            raise ValueError(f"slice_range は (start, end) 形式で指定してください: {slice_range}")

        start, end = int(slice_range[0]), int(slice_range[1])
        z_size = seed_volume.shape[0]
        start = max(1, start)
        end = min(z_size, end)
        if start > end:
            return

        # 1-based -> 0-based
        keep_mask[start - 1:end] = True

    _mark_slice_range(upper_slice_range)
    _mark_slice_range(lower_slice_range)

    # どちらも未指定ならそのまま返す
    if not np.any(keep_mask):
        return seed_volume

    filtered_seed = np.zeros_like(seed_volume)
    filtered_seed[keep_mask] = seed_volume[keep_mask]
    return filtered_seed


def graph_main_with_slice_ranges(
    input_path: str,
    upper_slice_range: tuple | None = None,
    lower_slice_range: tuple | None = None,
    inverse_volume: bool = True,
):
    merged_graph, seed_volume = graph_main(input_path, inverse_volume=inverse_volume)
    seed_volume = apply_seed_slice_ranges(
        seed_volume,
        upper_slice_range=upper_slice_range,
        lower_slice_range=lower_slice_range,
    )
    return merged_graph, seed_volume

In [114]:
import os

import numpy as np
import tifffile as tiff
from scipy import ndimage as ndi
from skimage import filters, morphology, segmentation, feature, util
import matplotlib.pyplot as plt

def save_colorized_labels_local(labels: np.ndarray, color_out: str):
    # ラベル画像をRGBカラーで保存する
    max_label = int(labels.max())
    if max_label == 0:
        rgb = np.zeros(labels.shape + (3,), dtype=np.uint8)
    else:
        cmap = plt.get_cmap("nipy_spectral", max_label + 1)
        normalized = labels.astype(np.float32) / max_label
        rgb = (cmap(normalized)[..., :3] * 255).astype(np.uint8)
        rgb[labels == 0] = 0
    tiff.imwrite(color_out, rgb, photometric="rgb")

def watershed_3d_tiff(volume: np.ndarray, seed_volume: np.ndarray, labels_out: str, connectivity: int = 6):
    # --- 1) 読み込み（3D）---
    vol = volume  # (Z, Y, X)
    if vol.ndim != 3:
        raise ValueError(f"3Dボリュームを想定していますが、形状が {vol.shape} です。")
    # print(f"入力ボリュームの形状: {vol.shape}, データ型: {vol.dtype}")
    vol = util.img_as_float(vol)

    # --- 2) 前処理 ---
    vol_smooth = filters.gaussian(vol, sigma=1.0, preserve_range=True)

    # --- 3) 前景抽出 ---
    bw = vol > 0

    # --- 4) 距離変換（3D） & マーカー ---
    dist = ndi.distance_transform_edt(bw)
    dist_out = os.path.join(os.path.dirname(labels_out), f"{os.path.splitext(os.path.basename(labels_out))[0]}_dist.tif")
    tiff.imwrite(dist_out, dist.astype(np.float32))

    if seed_volume is None:
        coords = feature.peak_local_max(
            dist,
            labels=bw,
            footprint=np.ones((3, 3, 3), dtype=bool),
            exclude_border=False
        )
        markers = np.zeros_like(dist, dtype=np.int32)
        if len(coords) > 0:
            markers[tuple(coords.T)] = np.arange(1, len(coords) + 1, dtype=np.int32)
    else:
        markers = seed_volume  # (Z, Y, X)

    # --- 5) 3D watershed ---
    labels = segmentation.watershed(
        -dist,
        markers=markers,
        mask=bw,
        connectivity=connectivity,
        watershed_line=False
    )

    # --- 6) 保存 ---
    print(f"ラベル数: {labels.max()}")
    os.makedirs(os.path.dirname(labels_out), exist_ok=True)
    tiff.imwrite(labels_out, labels.astype(np.uint16))
    # print(f"Saved: {labels_out}")
    # print(f"形状: {labels.shape}, データ型: {labels.dtype}")
    for i in range(1, labels.max() + 1):
        # print(f"ラベル {i}: ボクセル数 = {np.sum(labels == i)}")
        pass
    base, ext = os.path.splitext(labels_out)
    color_out = base + '_color' + ext
    save_colorized_labels_local(labels, color_out)
    print(f"Saved colorized: {color_out}")

# 使い方例
# watershed_3d_tiff(
#     volume=tiff.imread("./study/Watershed/Data/Input/20260313_test/volume_1.tif"),
#     seed_volume=tiff.imread("./study/Watershed/Data/Input/20260313_test/label_map_1_annotate.tif"),
#     labels_out="./study/Watershed/Data/Output/20260313/watershed_volume_1.tif"
# )

In [115]:
def plofile_slice(input_path: str):
    # X軸: スライス番号、Y軸: 面積と連結成分数の2軸で表示
    volume = load_volume(input_path)

    slice_indices = []
    slice_areas = []
    component_counts = []

    structure = ndi.generate_binary_structure(rank=2, connectivity=2)
    for index, curr_slice in enumerate(volume):
        slice_index = index + 1
        _, labels_number = ndi.label(curr_slice, structure=structure)
        total_area = int(np.count_nonzero(curr_slice))

        slice_indices.append(slice_index)
        slice_areas.append(total_area)
        component_counts.append(int(labels_number))

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx()

    line1 = ax1.plot(
        slice_indices,
        slice_areas,
        color="tab:blue",
        marker="o",
        markersize=3,
        linewidth=1,
        label="Area per Slice",
    )
    line2 = ax2.plot(
        slice_indices,
        component_counts,
        color="tab:red",
        marker="x",
        markersize=3,
        linewidth=1,
        label="Connected Components",
    )

    ax1.set_xlabel("Slice Index")
    ax1.set_ylabel("Area", color="tab:blue")
    ax2.set_ylabel("Number of Connected Components", color="tab:red")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax1.grid(True)

    lines = line1 + line2
    labels = [line.get_label() for line in lines]
    ax1.legend(lines, labels, loc="upper right")
    plt.title(os.path.basename(input_path))
    plt.tight_layout()
    plt.show()


In [116]:
def slice_change(input_path: str):
    # coronal方向に変換して保存する
    volume = load_volume(input_path)
    coronal_volume = np.transpose(volume, (1, 0, 2))
    tiff.imwrite(r"D:\_study\ImageProcessing\study\Watershed\Data\output\20260609\coronal_volume.tif", 
                 coronal_volume.astype(np.uint16), dtype=np.uint16)

In [117]:
import numpy as np
from skimage import measure

def numpy_to_mesh(volume, level=0.5):
    """
    volume: 3D numpy array (binary or scalar)
    level: iso-surface threshold
    return:
        verts: 頂点 (N, 3)
        faces: 三角形インデックス (M, 3)
        normals: 法線
        values: スカラー値
    """
    verts, faces, normals, values = measure.marching_cubes(volume, level=level)
    return verts, faces, normals, values
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def plot_mesh(verts, faces):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    mesh = Poly3DCollection(verts[faces])
    mesh.set_edgecolor('k')
    mesh.set_alpha(0.7)

    ax.add_collection3d(mesh)

    ax.set_xlim(0, verts[:,0].max())
    ax.set_ylim(0, verts[:,1].max())
    ax.set_zlim(0, verts[:,2].max())

    plt.show()
if False:
    path = "D:\_study\ImageProcessing\study\Watershed\Data\Input\labeled_map_filled\label_map_8_filled.tif"
    volume = tiff.imread(path)
    verts, faces, normals, values = numpy_to_mesh(volume, level=0.5)
    plot_mesh(verts, faces)

<>:35: SyntaxWarning: invalid escape sequence '\_'
<>:35: SyntaxWarning: invalid escape sequence '\_'
C:\temp\ipykernel_31412\2842500439.py:35: SyntaxWarning: invalid escape sequence '\_'
  path = "D:\_study\ImageProcessing\study\Watershed\Data\Input\labeled_map_filled\label_map_8_filled.tif"


# rev3: PCA前歯/臼歯分割を使ったグラフseed作成

グラフ作成の前に、入力ボリュームのAxial投影へPCAを行い、PCA軸のうち画像Y方向に近い軸を前歯/臼歯の分割軸として使います。軸の符号は画像上側が小さいスコアになる向きにそろえ、画像上側を前歯、画像下側を臼歯として扱います。

前歯部と臼歯部では連結成分の面積スケールが異なるため、グラフ作成時のthresholdは `anterior_threshold` と `molar_threshold` で別々に指定できます。未指定の場合は従来どおり共通の `threshold` を使います。

処理の流れ:

1. 入力ボリュームからAxial投影を作る
2. 前景点群にPCAを行う
3. 画像Y方向に近いPCA軸で前歯部/臼歯部を分割する
4. 前歯部ボリューム、臼歯部ボリュームを作る
5. 前歯部、臼歯部それぞれに別thresholdでグラフ作成とエッジ削除を行う
6. 下から/上からの両方向でseedを作成する
7. 前歯seedと臼歯seedを重複しないラベル番号へ統合する
8. 統合seedで3D watershedを実行する


In [118]:
# rev3: PCAで画像上側=前歯、画像下側=臼歯として分割する補助関数

def pca_axes_xy(points_xy: np.ndarray):
    if len(points_xy) < 3:
        raise ValueError("PCAには3点以上が必要です。")
    center = points_xy.mean(axis=0)
    centered = points_xy - center
    covariance = np.cov(centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    return center, eigenvectors[:, order[0]], eigenvectors[:, order[1]], eigenvalues[order]


def axial_projection_points_xy(volume: np.ndarray):
    mask = volume > 0
    projection = np.any(mask, axis=0)
    ys, xs = np.nonzero(projection)
    if len(xs) < 20:
        raise ValueError("Axial投影の前景点が少なすぎます。")
    points_xy = np.column_stack([xs.astype(np.float64), ys.astype(np.float64)])
    return projection, points_xy


def split_volume_anterior_molar_by_pca(
    volume: np.ndarray,
    split_percentile: float = 50.0,
    ambiguous_band_px: float = 0.0,
):
    projection, points_xy = axial_projection_points_xy(volume)
    center_xy, pc1_xy, pc2_xy, eigenvalues = pca_axes_xy(points_xy)
    axes = [pc1_xy, pc2_xy]
    split_axis_index = int(np.argmax([abs(axis[1]) for axis in axes]))
    split_axis_xy = axes[split_axis_index].copy()
    along_arch_axis_xy = axes[1 - split_axis_index].copy()

    if split_axis_xy[1] < 0:
        split_axis_xy = -split_axis_xy

    scores = (points_xy - center_xy) @ split_axis_xy
    split_score = float(np.percentile(scores, split_percentile))

    height, width = projection.shape
    yy, xx = np.indices((height, width))
    grid_points = np.column_stack([xx.ravel().astype(np.float64), yy.ravel().astype(np.float64)])
    grid_scores = ((grid_points - center_xy) @ split_axis_xy).reshape(height, width)

    anterior_xy_mask = projection & (grid_scores <= split_score - ambiguous_band_px)
    molar_xy_mask = projection & (grid_scores >= split_score + ambiguous_band_px)
    if ambiguous_band_px <= 0:
        anterior_xy_mask = projection & (grid_scores <= split_score)
        molar_xy_mask = projection & (grid_scores > split_score)

    volume_mask = volume > 0
    anterior_volume = volume_mask & anterior_xy_mask[None, :, :]
    molar_volume = volume_mask & molar_xy_mask[None, :, :]

    split_label_volume = np.zeros_like(volume, dtype=np.uint8)
    split_label_volume[anterior_volume] = 1
    split_label_volume[molar_volume] = 2
    if ambiguous_band_px > 0:
        ambiguous_xy_mask = projection & ~(anterior_xy_mask | molar_xy_mask)
        split_label_volume[volume_mask & ambiguous_xy_mask[None, :, :]] = 3

    line_t = np.linspace(-max(height, width), max(height, width), 600)
    split_line_xy = center_xy + split_score * split_axis_xy + np.outer(line_t, along_arch_axis_xy)

    info = {
        "center_xy": center_xy.astype(float).tolist(),
        "pc1_xy": pc1_xy.astype(float).tolist(),
        "pc2_xy": pc2_xy.astype(float).tolist(),
        "eigenvalues_desc": eigenvalues.astype(float).tolist(),
        "split_axis_index": int(split_axis_index),
        "split_axis_xy": split_axis_xy.astype(float).tolist(),
        "along_arch_axis_xy": along_arch_axis_xy.astype(float).tolist(),
        "split_score": float(split_score),
        "split_percentile": float(split_percentile),
        "ambiguous_band_px": float(ambiguous_band_px),
        "label_definition": {"0": "background", "1": "anterior/image-upper", "2": "molar/image-lower", "3": "ambiguous"},
        "anterior_voxel_count": int(np.count_nonzero(anterior_volume)),
        "molar_voxel_count": int(np.count_nonzero(molar_volume)),
    }
    return anterior_volume, molar_volume, split_label_volume, split_line_xy, info


def make_anterior_molar_split_overlay_volume(
    volume: np.ndarray,
    split_label_volume: np.ndarray,
    split_line_xy: np.ndarray,
    line_radius_px: int = 2,
):
    foreground = volume > 0
    base = np.zeros_like(volume, dtype=np.uint8)
    if np.any(foreground):
        foreground_values = volume[foreground].astype(np.float32)
        scale_max = float(np.percentile(foreground_values, 99))
        if scale_max <= 0:
            scale_max = float(np.max(foreground_values))
        base[foreground] = np.clip(volume[foreground].astype(np.float32) / scale_max * 180, 20, 180).astype(np.uint8)

    overlay = np.repeat(base[..., None], 3, axis=-1)

    height, width = volume.shape[1], volume.shape[2]
    line_points = np.rint(split_line_xy).astype(np.int32)
    for x, y in line_points:
        if not (0 <= x < width and 0 <= y < height):
            continue
        y0 = max(0, y - line_radius_px)
        y1 = min(height, y + line_radius_px + 1)
        x0 = max(0, x - line_radius_px)
        x1 = min(width, x + line_radius_px + 1)
        overlay[:, y0:y1, x0:x1] = (255, 40, 40)

    overlay[volume <= 0] = 0
    return overlay


def summarize_region_boundary_slices_from_split_labels(split_label_volume: np.ndarray):
    # label 1: anterior(image-upper), label 2: molar(image-lower), label 3: ambiguous(boundary)
    # コンソール表示用に、各ラベルが存在するZスライス(1-based)を要約する

    def _slice_indices_for_label(label_value: int):
        present = np.any(split_label_volume == int(label_value), axis=(1, 2))
        return [int(index + 1) for index in np.where(present)[0]]

    def _to_ranges(indices_1based: list[int]):
        if len(indices_1based) == 0:
            return []
        ranges = []
        start = int(indices_1based[0])
        prev = int(indices_1based[0])
        for current in indices_1based[1:]:
            current = int(current)
            if current == prev + 1:
                prev = current
                continue
            ranges.append((start, prev))
            start = current
            prev = current
        ranges.append((start, prev))
        return ranges

    anterior_slices = _slice_indices_for_label(1)
    molar_slices = _slice_indices_for_label(2)
    boundary_slices = _slice_indices_for_label(3)

    return {
        "anterior_slices_1based": anterior_slices,
        "molar_slices_1based": molar_slices,
        "boundary_slices_1based": boundary_slices,
        "anterior_slice_ranges_1based": _to_ranges(anterior_slices),
        "molar_slice_ranges_1based": _to_ranges(molar_slices),
        "boundary_slice_ranges_1based": _to_ranges(boundary_slices),
    }


In [119]:
# rev3: 前歯/臼歯ごとにグラフseedを作り、上下方向の結果を統合してwatershedする
import json


def relabel_seed_volume(seed_volume: np.ndarray, start_label: int = 1):
    output = np.zeros_like(seed_volume, dtype=np.uint16)
    next_label = int(start_label)
    for label_value in np.unique(seed_volume):
        label_value = int(label_value)
        if label_value == 0:
            continue
        output[seed_volume == label_value] = next_label
        next_label += 1
    return output, next_label


def create_execution_slice_mask(z_size: int, execution_slice_range: tuple | None = None):
    mask = np.ones(z_size, dtype=bool)
    normalized_range = None
    if execution_slice_range is None:
        return mask, normalized_range
    if len(execution_slice_range) != 2:
        raise ValueError(f"execution_slice_range は (start, end) 形式で指定してください: {execution_slice_range}")

    start, end = int(execution_slice_range[0]), int(execution_slice_range[1])
    start = max(1, start)
    end = min(z_size, end)
    if start > end:
        raise ValueError(f"有効なスライス範囲がありません: {execution_slice_range}")

    mask[:] = False
    mask[start - 1:end] = True
    normalized_range = (start, end)
    return mask, normalized_range


def plot_graph_for_slice_range(
    G: nx.Graph,
    slice_range: tuple | None = None,
    output_path: str | None = None,
    title: str | None = None,
    show: bool = False,
):
    real_nodes = [
        node
        for node in G.nodes
        if isinstance(node[1], (int, np.integer))
    ]
    if slice_range is not None:
        start, end = int(slice_range[0]), int(slice_range[1])
        real_nodes = [node for node in real_nodes if start <= node[0] <= end]

    slices_with_real_nodes = sorted({node[0] for node in real_nodes})
    if not slices_with_real_nodes:
        return None

    draw_nodes = set(real_nodes)
    draw_nodes.update((slice_index, "slice_index") for slice_index in slices_with_real_nodes if (slice_index, "slice_index") in G)
    draw_graph = G.subgraph(draw_nodes).copy()

    min_slice = min(slices_with_real_nodes)
    max_slice = max(slices_with_real_nodes)
    label_values = [int(node[1]) for node in real_nodes]
    max_label = max(label_values) if label_values else 1

    pos = {}
    for node in draw_graph.nodes:
        slice_index, label = node
        if label == "slice_index":
            pos[node] = (0, slice_index)
        else:
            pos[node] = (int(label), slice_index)

    height = max(8, min(80, (max_slice - min_slice + 1) * 0.35))
    width = max(10, min(36, max_label * 0.9 + 3))
    plt.figure(figsize=(width, height))

    labels = {}
    for node in draw_graph.nodes:
        if node[1] == "slice_index":
            labels[node] = f"slice {node[0]}"
        else:
            labels[node] = f"{draw_graph.nodes[node].get('area', '')}"

    node_colors = [
        "red" if draw_graph.nodes[node].get("seed", False) else "skyblue"
        for node in draw_graph.nodes
    ]
    edge_colors = [
        draw_graph[u][v].get("color", "black")
        for u, v in draw_graph.edges
    ]

    nx.draw(
        draw_graph,
        pos=pos,
        with_labels=True,
        labels=labels,
        node_color=node_colors,
        edge_color=edge_colors,
        node_size=450,
        font_size=7,
    )
    if title is not None:
        plt.title(title)
    plt.ylim(min_slice - 1, max_slice + 1)
    plt.tight_layout()

    if output_path is not None:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close()
    return output_path


def graph_seed_volume_bidirectional(
    volume: np.ndarray,
    threshold: int = 1000,
    labels_connectivity: int = 2,
    inverse_volume: bool = True,
    direction_mode: str | None = None,
):
    if direction_mode is None:
        direction_mode = "bidirectional" if inverse_volume else "forward_only"
    direction_mode = str(direction_mode).lower()
    valid_modes = {"forward_only", "reverse_only", "bidirectional"}
    if direction_mode not in valid_modes:
        raise ValueError(f"direction_mode は {sorted(valid_modes)} のいずれかで指定してください: {direction_mode}")

    use_forward = direction_mode in {"forward_only", "bidirectional"}
    use_reverse = direction_mode in {"reverse_only", "bidirectional"}

    graph_forward, labeled_volume_forward = create_graph(
        volume,
        threshold=threshold,
        labels_connectivity=labels_connectivity,
    )
    remove_edge(graph_forward)

    graph_reverse = None
    labeled_volume_reverse = None
    merged_graph = graph_forward

    if use_reverse:
        volume_reverse = volume[::-1]
        graph_reverse, labeled_volume_reverse = create_graph(
            volume_reverse,
            threshold=threshold,
            labels_connectivity=labels_connectivity,
        )
        remove_edge(graph_reverse)
        if use_forward:
            merged_graph = merge_bidirectional_graph(graph_forward, graph_reverse, volume.shape[0])
        else:
            merged_graph = nx.Graph()
            for node, attrs in graph_reverse.nodes(data=True):
                mapped_node = (volume.shape[0] - node[0] + 1, node[1])
                merged_graph.add_node(mapped_node, **dict(attrs))
            for node_u, node_v, attrs in graph_reverse.edges(data=True):
                mapped_u = (volume.shape[0] - node_u[0] + 1, node_u[1])
                mapped_v = (volume.shape[0] - node_v[0] + 1, node_v[1])
                merged_graph.add_edge(mapped_u, mapped_v, color=attrs.get("color", "black"))
    elif not use_forward:
        raise ValueError("有効な方向モードが選択されていません。")

    non_bifurcating_components = detect_non_bifurcating_components(merged_graph)
    added_seed_nodes = add_largest_area_seed_in_components(merged_graph, non_bifurcating_components)

    for node in added_seed_nodes:
        if use_forward and node in graph_forward:
            graph_forward.nodes[node]["seed"] = True
        if use_reverse and graph_reverse is not None:
            reverse_node = (volume.shape[0] - node[0] + 1, node[1])
            if reverse_node in graph_reverse:
                graph_reverse.nodes[reverse_node]["seed"] = True

    seed_forward = create_seed_for_watershed(graph_forward, labeled_volume_forward) if use_forward else None
    seed_volume = seed_forward if seed_forward is not None else np.zeros_like(volume, dtype=np.uint16)

    if use_reverse and graph_reverse is not None and labeled_volume_reverse is not None:
        seed_reverse = create_seed_for_watershed(graph_reverse, labeled_volume_reverse)
        seed_reverse = seed_reverse[::-1]
        if seed_forward is not None:
            seed_volume = np.where(seed_forward != 0, seed_forward, seed_reverse)
        else:
            seed_volume = seed_reverse

    return merged_graph, seed_volume, {
        "threshold": int(threshold),
        "inverse_volume": bool(inverse_volume),
        "direction_mode": direction_mode,
        "node_count": int(merged_graph.number_of_nodes()),
        "edge_count": int(merged_graph.number_of_edges()),
        "seed_voxel_count": int(np.count_nonzero(seed_volume)),
        "added_seed_node_count": int(len(added_seed_nodes)),
        "non_bifurcating_component_count": int(len(non_bifurcating_components)),
    }


def graph_main_anterior_molar_pca_rev3(
    input_path: str,
    output_dir: str,
    threshold: int = 1000,
    anterior_threshold: int | None = None,
    molar_threshold: int | None = None,
    labels_connectivity: int = 2,
    split_percentile: float = 50.0,
    ambiguous_band_px: float = 0.0,
    inverse_volume: bool = True,
    direction_mode: str | None = None,
    execution_slice_range: tuple | None = None,
    plot_execution_slice_graphs: bool = False,
    show_graph_plots: bool = False,
    run_watershed: bool = True,
):
    os.makedirs(output_dir, exist_ok=True)
    original_volume = load_volume(input_path)
    execution_slice_mask, normalized_slice_range = create_execution_slice_mask(
        original_volume.shape[0],
        execution_slice_range=execution_slice_range,
    )
    volume = np.where(execution_slice_mask[:, None, None], original_volume, 0).astype(original_volume.dtype)
    volume_mask = volume > 0
    if not np.any(volume_mask):
        raise ValueError(f"指定したスライス範囲に前景ボクセルがありません: {normalized_slice_range}")

    if anterior_threshold is None:
        anterior_threshold = threshold
    if molar_threshold is None:
        molar_threshold = threshold

    anterior_volume_mask, molar_volume_mask, split_label_volume, split_line_xy, pca_info = split_volume_anterior_molar_by_pca(
        volume,
        split_percentile=split_percentile,
        ambiguous_band_px=ambiguous_band_px,
    )
    tiff.imwrite(os.path.join(output_dir, "anterior_molar_pca_split_labels.tif"), split_label_volume.astype(np.uint8))

    split_overlay_volume = make_anterior_molar_split_overlay_volume(
        volume=volume,
        split_label_volume=split_label_volume,
        split_line_xy=split_line_xy,
        line_radius_px=2,
    )
    split_overlay_path = os.path.join(output_dir, "anterior_molar_pca_split_overlay.tif")
    tiff.imwrite(split_overlay_path, split_overlay_volume, photometric="rgb")
    region_boundary_slice_summary = summarize_region_boundary_slices_from_split_labels(split_label_volume)

    combined_seed = np.zeros_like(volume, dtype=np.uint16)
    next_seed_label = 1
    region_infos = {}
    graph_plot_paths = {}
    region_specs = {
        "anterior": {
            "mask": anterior_volume_mask,
            "threshold": int(anterior_threshold),
        },
        "molar": {
            "mask": molar_volume_mask,
            "threshold": int(molar_threshold),
        },
    }

    for region_name, region_spec in region_specs.items():
        region_mask = region_spec["mask"]
        region_threshold = region_spec["threshold"]
        region_volume = np.where(volume_mask & region_mask, volume, 0).astype(volume.dtype)
        graph, seed_volume, graph_info = graph_seed_volume_bidirectional(
            region_volume,
            threshold=region_threshold,
            labels_connectivity=labels_connectivity,
            inverse_volume=inverse_volume,
            direction_mode=direction_mode,
        )
        if plot_execution_slice_graphs:
            graph_plot_path = os.path.join(output_dir, f"{region_name}_execution_slice_graph.png")
            saved_graph_path = plot_graph_for_slice_range(
                graph,
                slice_range=normalized_slice_range,
                output_path=graph_plot_path,
                title=f"{region_name} graph slices {normalized_slice_range or 'all'}",
                show=show_graph_plots,
            )
            graph_plot_paths[region_name] = saved_graph_path
            graph_info["graph_plot_path"] = saved_graph_path

        relabeled_seed, next_seed_label = relabel_seed_volume(seed_volume, start_label=next_seed_label)
        combined_seed = np.where(combined_seed != 0, combined_seed, relabeled_seed).astype(np.uint16)
        tiff.imwrite(os.path.join(output_dir, f"{region_name}_seed_volume.tif"), relabeled_seed.astype(np.uint16))
        region_infos[region_name] = graph_info

    combined_seed[~volume_mask] = 0
    seed_path = os.path.join(output_dir, "combined_anterior_molar_seed_volume.tif")
    tiff.imwrite(seed_path, combined_seed.astype(np.uint16))

    watershed_output_path = None
    if run_watershed:
        watershed_output_path = os.path.join(output_dir, "watershed_anterior_molar_rev3.tif")
        watershed_3d_tiff(
            volume=volume,
            seed_volume=combined_seed,
            labels_out=watershed_output_path,
        )

    info = {
        "input_path": input_path,
        "threshold": int(threshold),
        "anterior_threshold": int(anterior_threshold),
        "molar_threshold": int(molar_threshold),
        "labels_connectivity": int(labels_connectivity),
        "inverse_volume": bool(inverse_volume),
        "direction_mode": str(direction_mode) if direction_mode is not None else ("bidirectional" if inverse_volume else "forward_only"),
        "execution_slice_range": normalized_slice_range,
        "plot_execution_slice_graphs": bool(plot_execution_slice_graphs),
        "graph_plot_paths": graph_plot_paths,
        "split_overlay_path": split_overlay_path,
        "region_boundary_slice_summary": region_boundary_slice_summary,
        "pca_split": pca_info,
        "region_graphs": region_infos,
        "seed_path": seed_path,
        "watershed_output_path": watershed_output_path,
        "workflow": "PCA anterior/molar split -> directional graph seeds per region -> seed merge -> watershed",
    }
    info_path = os.path.join(output_dir, "rev3_pca_graph_watershed_info.json")
    with open(info_path, "w", encoding="utf-8") as file:
        json.dump(info, file, ensure_ascii=False, indent=2)

    return combined_seed, info


In [120]:
# rev3 実行例
# 前歯部と臼歯部でthresholdを分けて指定できます。
# direction_mode は "reverse_only"(逆側のみ) / "bidirectional"(順+逆) を指定できます。

def run_rev3_example():
    input_path = r"D:\_study\ImageProcessing\study\Watershed\Data\Output\rotated_volume.tif"
    input_path = r"D:\_study\ImageProcessing\study\Watershed\Data\Output\test\_label_map_8_filled_aligned.tif"
    output_dir = r"D:\_study\ImageProcessing\study\Watershed\Data\Output\test"
    direction_mode = "reverse_only"  # "reverse_only" or "bidirectional"
    combined_seed, info = graph_main_anterior_molar_pca_rev3(
        input_path=input_path,
        output_dir=output_dir,
        threshold=1000,
        anterior_threshold=1000,
        molar_threshold=2000,
        labels_connectivity=2,
        split_percentile=50.0,
        ambiguous_band_px=0.0,
        inverse_volume=True,
        direction_mode=direction_mode,
        # execution_slice_range=(160, 400),  # 例: (120, 260)。Noneなら全スライス。
        # execution_slice_range=(360, 600),
        execution_slice_range=None,
        plot_execution_slice_graphs=False,
        show_graph_plots=False,
        run_watershed=False,
    )
    print(f"anterior threshold: {info['anterior_threshold']}")
    print(f"molar threshold: {info['molar_threshold']}")
    print(f"inverse volume: {info['inverse_volume']}")
    print(f"direction mode: {info['direction_mode']}")
    print(f"execution slice range: {info['execution_slice_range']}")
    print(f"anterior slices (1-based): {info['region_boundary_slice_summary']['anterior_slices_1based']}")
    print(f"molar slices (1-based): {info['region_boundary_slice_summary']['molar_slices_1based']}")
    print(f"boundary slices (1-based): {info['region_boundary_slice_summary']['boundary_slices_1based']}")
    print(f"anterior slice ranges: {info['region_boundary_slice_summary']['anterior_slice_ranges_1based']}")
    print(f"molar slice ranges: {info['region_boundary_slice_summary']['molar_slice_ranges_1based']}")
    print(f"boundary slice ranges: {info['region_boundary_slice_summary']['boundary_slice_ranges_1based']}")
    print(f"graph plots: {info['graph_plot_paths']}")
    print(f"seed labels: {combined_seed.max()}")
    print(f"outputs: {output_dir}")
    return combined_seed, info

# 実行する場合:
combined_seed, info = run_rev3_example()


anterior threshold: 1000
molar threshold: 2000
inverse volume: True
direction mode: reverse_only
execution slice range: None
anterior slices (1-based): [204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 37